In [1]:
%pip install requests beautifulsoup4 pandas

Note: you may need to restart the kernel to use updated packages.


In [2]:
import requests
from bs4 import BeautifulSoup
import pandas as pd

url = "https://books.toscrape.com/"

response = requests.get(url, timeout=15)
response.raise_for_status()

soup = BeautifulSoup(response.text, "html.parser")

print("Status Code:", response.status_code)
print("Page Title:", soup.title.text)

Status Code: 200
Page Title: 
    All products | Books to Scrape - Sandbox



In [3]:
books = soup.select("article.product_pod")

for book in books[:5]:
    title = book.select_one("h3 a")["title"]
    price = book.select_one(".price_color").text
    rating = book.select_one("p.star-rating")["class"][1]

    print("Title:", title)
    print("Price:", price)
    print("Rating:", rating)
    print("-------------------")

Title: A Light in the Attic
Price: Â£51.77
Rating: Three
-------------------
Title: Tipping the Velvet
Price: Â£53.74
Rating: One
-------------------
Title: Soumission
Price: Â£50.10
Rating: One
-------------------
Title: Sharp Objects
Price: Â£47.82
Rating: Four
-------------------
Title: Sapiens: A Brief History of Humankind
Price: Â£54.23
Rating: Five
-------------------


In [4]:
from urllib.parse import urljoin

def scrape_books(min_rating, max_price):
    books_data = []
    ratings = {
        "One": 1, "Two": 2, "Three": 3,
        "Four": 4, "Five": 5
    }

    for page in range(1, 51):
        url = f"https://books.toscrape.com/catalogue/page-{page}.html"
        response = requests.get(url, timeout=15)
        response.raise_for_status()
        soup = BeautifulSoup(response.text, "html.parser")

        for book in soup.select("article.product_pod"):
            title = book.select_one("h3 a")["title"]
            price = float(
                book.select_one(".price_color").text.replace("£", "")
            )
            rating_class = book.select_one("p.star-rating")["class"][1]
            rating = ratings[rating_class]

            if rating >= min_rating and price <= max_price:
                books_data.append({
                    "Title": title,
                    "Price (£)": price,
                    "Rating": rating
                })

    return pd.DataFrame(
        books_data,
        columns=["Title", "Price (£)", "Rating"]
    )

In [7]:
def scrape_books(min_rating, max_price):
    books_data = []
    ratings = {
        "One": 1, "Two": 2, "Three": 3,
        "Four": 4, "Five": 5
    }

    for page in range(1, 51):
        url = f"https://books.toscrape.com/catalogue/page-{page}.html"
        response = requests.get(url, timeout=15)
        response.raise_for_status()
        soup = BeautifulSoup(response.text, "html.parser")

        for book in soup.select("article.product_pod"):
            price = float(
                book.select_one(".price_color").text.replace("£", "").replace("Â", "")
            )
            rating_class = book.select_one("p.star-rating")["class"][1]
            rating = ratings[rating_class]

            if rating >= min_rating and price <= max_price:
                link = book.select_one("h3 a")["href"]
                book_url = urljoin(url, link)

                detail_response = requests.get(book_url, timeout=15)
                detail_response.raise_for_status()
                detail = BeautifulSoup(
                    detail_response.text, "html.parser"
                )

                info = {}
                for row in detail.select("table.table-striped tr"):
                    key = row.select_one("th").get_text(strip=True)
                    value = row.select_one("td").get_text(strip=True)
                    info[key] = value

                breadcrumbs = detail.select("ul.breadcrumb li a")
                genre = breadcrumbs[2].get_text(strip=True)

                description_tag = detail.select_one(
                    "#product_description + p"
                )

                books_data.append({
                    "UPC": info.get("UPC"),
                    "Title": detail.select_one(
                        ".product_main h1"
                    ).get_text(strip=True),
                    "Price (£)": price,
                    "Rating": rating,
                    "Genre": genre,
                    "Availability": info.get("Availability"),
                    "Description": (
                        description_tag.get_text(strip=True)
                        if description_tag else ""
                    )
                })

    return pd.DataFrame(books_data, columns=[
        "UPC", "Title", "Price (£)", "Rating",
        "Genre", "Availability", "Description"
    ])

In [ ]:
df_books = scrape_books(4.0, 20)

print("Number of books found:", len(df_books))

df_books.head(10)

In [ ]:
df_books.to_csv("books_scraped.csv", index=False)

print("CSV file saved successfully!")
print("Total books:", len(df_books))

In [ ]:
print("Total books:", len(df_books))
print("Maximum price:", df_books["Price (£)"].max())
print("Minimum rating:", df_books["Rating"].min())
print("Missing values:")
print(df_books.isnull().sum())